In [1]:
import pandas as pd
import numpy as np
import torch
import lightning as L
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.metrics import QuantileLoss
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("data/processed/features_panel.csv", index_col="Date")
df.index = pd.to_datetime(df.index)

print("Loaded:", df.shape)
print("Date range:", df.index[0], "to", df.index[-1])
print("GPU available:", torch.cuda.is_available())
print("Torch version:", torch.__version__)
print("Lightning version:", L.__version__)

Loaded: (91, 77)
Date range: 2024-01-15 00:00:00 to 2024-05-31 00:00:00
GPU available: False
Torch version: 2.11.0+cpu
Lightning version: 2.2.0


In [2]:
# define surface node columns
node_cols = [c for c in df.columns if any(
    c == f"{opt}_M{m}_D{d}"
    for opt in ["CE","PE"]
    for m in range(1,6)
    for d in range(1,5)
)]

# define feature columns
feature_cols = [c for c in df.columns if c not in node_cols]

print(f"Surface nodes: {len(node_cols)}")
print(f"Feature columns: {len(feature_cols)}")
print(f"Node columns: {node_cols}")
print(f"Feature columns: {feature_cols}")

Surface nodes: 40
Feature columns: 37
Node columns: ['CE_M1_D1', 'CE_M1_D2', 'CE_M1_D3', 'CE_M1_D4', 'CE_M2_D1', 'CE_M2_D2', 'CE_M2_D3', 'CE_M2_D4', 'CE_M3_D1', 'CE_M3_D2', 'CE_M3_D3', 'CE_M3_D4', 'CE_M4_D1', 'CE_M4_D2', 'CE_M4_D3', 'CE_M4_D4', 'CE_M5_D1', 'CE_M5_D2', 'CE_M5_D3', 'CE_M5_D4', 'PE_M1_D1', 'PE_M1_D2', 'PE_M1_D3', 'PE_M1_D4', 'PE_M2_D1', 'PE_M2_D2', 'PE_M2_D3', 'PE_M2_D4', 'PE_M3_D1', 'PE_M3_D2', 'PE_M3_D3', 'PE_M3_D4', 'PE_M4_D1', 'PE_M4_D2', 'PE_M4_D3', 'PE_M4_D4', 'PE_M5_D1', 'PE_M5_D2', 'PE_M5_D3', 'PE_M5_D4']
Feature columns: ['Spot', 'VIX', 'ATM_IV', 'ATM_IV_1d_chg', 'ATM_IV_5d_chg', 'Skew_D1', 'Skew_D2', 'TermSlope_CE', 'TermSlope_PE', 'SmileWidth_CE', 'SmileWidth_PE', 'Nifty_Return', 'Nifty_5d_Return', 'RealVol_5d', 'RealVol_10d', 'VIX_1d_chg', 'VIX_5d_chg', 'IV_Premium', 'OI_CE', 'OI_PE', 'PCR', 'OI_CE_chg', 'OI_PE_chg', 'DayOfWeek', 'IsExpiry', 'DaysSinceExpiry', 'Month', 'ATM_IV_lag1', 'ATM_IV_lag5', 'Skew_D1_lag1', 'Skew_D1_lag5', 'TermSlope_CE_lag1', 'TermSlop

In [3]:
df_reset = df.reset_index()
df_long = df_reset.melt(
    id_vars   = ["Date"] + feature_cols,
    value_vars = node_cols,
    var_name  = "Node",
    value_name = "IV"
)

df_long = df_long.sort_values(["Node","Date"]).reset_index(drop=True)
df_long["TimeIdx"] = df_long.groupby("Node")["Date"].rank(method="dense").astype(int) - 1
df_long["GroupID"] = df_long["Node"]

# convert categorical columns to string
for col in ["DayOfWeek","Month","IsExpiry"]:
    df_long[col] = df_long[col].astype(str)

print("Long format shape:", df_long.shape)
print("Expected:", 91*40, "rows")
print("\nSample:")
print(df_long[["Date","Node","GroupID","TimeIdx","IV"]].head(8))
print("\nUnique nodes:", df_long["Node"].nunique())
print("Unique time steps:", df_long["TimeIdx"].nunique())

Long format shape: (3640, 42)
Expected: 3640 rows

Sample:
        Date      Node   GroupID  TimeIdx        IV
0 2024-01-15  CE_M1_D1  CE_M1_D1        0  0.259455
1 2024-01-16  CE_M1_D1  CE_M1_D1        1  0.263250
2 2024-01-17  CE_M1_D1  CE_M1_D1        2  0.296413
3 2024-01-18  CE_M1_D1  CE_M1_D1        3  0.256414
4 2024-01-19  CE_M1_D1  CE_M1_D1        4  0.236586
5 2024-01-23  CE_M1_D1  CE_M1_D1        5  0.188142
6 2024-01-24  CE_M1_D1  CE_M1_D1        6  0.493781
7 2024-01-25  CE_M1_D1  CE_M1_D1        7  0.232659

Unique nodes: 40
Unique time steps: 91


In [4]:
max_time_idx = df_long["TimeIdx"].max()
train_cutoff = int(max_time_idx * 0.80)
val_cutoff   = int(max_time_idx * 0.90)

print(f"Total time steps: {max_time_idx}")
print(f"Train cutoff:     {train_cutoff}  (days 0–{train_cutoff})")
print(f"Val cutoff:       {val_cutoff}  (days {train_cutoff+1}–{val_cutoff})")
print(f"Test:             days {val_cutoff+1}–{max_time_idx}")

train_df = df_long[df_long["TimeIdx"] <= train_cutoff].copy()
val_df   = df_long[df_long["TimeIdx"] <= val_cutoff].copy()
test_df  = df_long.copy()

print(f"\nTrain rows: {len(train_df)}")
print(f"Val rows:   {len(val_df)}")
print(f"Test rows:  {len(test_df)}")

Total time steps: 90
Train cutoff:     72  (days 0–72)
Val cutoff:       81  (days 73–81)
Test:             days 82–90

Train rows: 2920
Val rows:   3280
Test rows:  3640


In [5]:
ENCODER_LENGTH = 20
DECODER_LENGTH = 5

time_varying_features = [
    "ATM_IV", "ATM_IV_1d_chg", "ATM_IV_5d_chg",
    "Skew_D1", "Skew_D2", "TermSlope_CE", "TermSlope_PE",
    "SmileWidth_CE", "SmileWidth_PE",
    "Nifty_Return", "Nifty_5d_Return",
    "RealVol_5d", "RealVol_10d",
    "VIX", "VIX_1d_chg", "VIX_5d_chg",
    "IV_Premium", "PCR", "OI_CE_chg", "OI_PE_chg",
    "ATM_IV_lag1", "ATM_IV_lag5",
    "Skew_D1_lag1", "Skew_D1_lag5",
    "PCR_lag1", "PCR_lag5",
    "VIX_lag1", "VIX_lag5"
]

training_dataset = TimeSeriesDataSet(
    train_df,
    time_idx                        = "TimeIdx",
    target                          = "IV",
    group_ids                       = ["GroupID"],
    max_encoder_length              = ENCODER_LENGTH,
    max_prediction_length           = DECODER_LENGTH,
    time_varying_known_categoricals = ["DayOfWeek","Month","IsExpiry"],
    time_varying_unknown_reals      = time_varying_features + ["IV"],
    target_normalizer               = None,
    add_relative_time_idx           = True,
    add_target_scales               = True,
    add_encoder_length              = True,
)

print("Training dataset created successfully.")
print("Number of samples:", len(training_dataset))

Training dataset created successfully.
Number of samples: 1960


In [6]:
train_loader = training_dataset.to_dataloader(
    train      = True,
    batch_size = 32,
    num_workers= 0
)

val_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset,
    val_df,
    predict            = True,
    stop_randomization = True
)

val_loader = val_dataset.to_dataloader(
    train      = False,
    batch_size = 32,
    num_workers= 0
)

print("Dataloaders created.")
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))

Dataloaders created.
Train batches: 61
Val batches: 2


In [7]:
tft = TemporalFusionTransformer.from_dataset(
    training_dataset,
    learning_rate          = 0.01,
    hidden_size            = 32,
    attention_head_size    = 2,
    dropout                = 0.1,
    hidden_continuous_size = 16,
    loss                   = QuantileLoss([0.1, 0.5, 0.9]),
    log_interval           = 10,
    reduce_on_plateau_patience = 4,
)

print("TFT model created.")
print(f"Number of parameters: {sum(p.numel() for p in tft.parameters()):,}")

TFT model created.
Number of parameters: 136,642


In [8]:
from lightning.pytorch.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor  = "val_loss",
    patience = 10,
    mode     = "min"
)

trainer = L.Trainer(
    max_epochs          = 50,
    accelerator         = "cpu",
    gradient_clip_val   = 0.1,
    callbacks           = [early_stop],
    enable_progress_bar = True,
    logger              = False,
    enable_checkpointing= False
)

print("Starting training — this will take 10–20 minutes on CPU.")
print("You will see epoch progress below. Let it run.\n")

trainer.fit(
    tft,
    train_dataloaders = train_loader,
    val_dataloaders   = val_loader
)

print("\nTraining complete.")
print("Best val loss:", trainer.callback_metrics.get("val_loss"))

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


Starting training — this will take 10–20 minutes on CPU.
You will see epoch progress below. Let it run.




   | Name                               | Type                            | Params
----------------------------------------------------------------------------------------
0  | loss                               | QuantileLoss                    | 0     
1  | logging_metrics                    | ModuleList                      | 0     
2  | input_embeddings                   | MultiEmbedding                  | 42    
3  | prescalers                         | ModuleDict                      | 1.1 K 
4  | static_variable_selection          | VariableSelectionNetwork        | 5.7 K 
5  | encoder_variable_selection         | VariableSelectionNetwork        | 75.1 K
6  | decoder_variable_selection         | VariableSelectionNetwork        | 2.4 K 
7  | static_context_variable_selection  | GatedResidualNetwork            | 4.3 K 
8  | static_context_initial_hidden_lstm | GatedResidualNetwork            | 4.3 K 
9  | static_context_initial_cell_lstm   | GatedResidualNetwork            | 4.3 

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]


Training complete.
Best val loss: tensor(0.0153)


In [10]:
import pickle
import os

os.makedirs("models", exist_ok=True)

# save model weights
torch.save(tft.state_dict(), "models/tft_v1.pt")

# save dataset parameters
with open("models/training_dataset_params.pkl","wb") as f:
    pickle.dump(training_dataset.get_parameters(), f)

# save training dataset itself for predictions later
with open("models/training_dataset.pkl","wb") as f:
    pickle.dump(training_dataset, f)

print("Model saved successfully.")
print("Files in models/:", os.listdir("models"))

Model saved successfully.
Files in models/: ['tft_v1.pt', 'tft_v1_full.pt', 'training_dataset.pkl', 'training_dataset_params.pkl']


In [11]:
test_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset,
    test_df,
    predict            = True,
    stop_randomization = True
)

test_loader = test_dataset.to_dataloader(
    train      = False,
    batch_size = 32,
    num_workers= 0
)

predictions = tft.predict(
    test_loader,
    mode       = "quantiles",
    return_x   = True
)

print("Predictions generated.")
print("Output shape:", predictions.output.shape)
print("\nShape means: (samples, forecast_horizon, quantiles)")
print("Quantiles: [P10, P50, P90]")
print("\nSample — first surface node, all 5 forecast days:")
print("P10  |  P50  |  P90")
for i in range(predictions.output.shape[1]):
    p10 = predictions.output[0, i, 0].item()
    p50 = predictions.output[0, i, 1].item()
    p90 = predictions.output[0, i, 2].item()
    print(f"{p10:.4f} | {p50:.4f} | {p90:.4f}")

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
Missing logger folder: c:\Users\Anik\Desktop\vol_surface_project\lightning_logs


Predictions generated.
Output shape: torch.Size([40, 5, 3])

Shape means: (samples, forecast_horizon, quantiles)
Quantiles: [P10, P50, P90]

Sample — first surface node, all 5 forecast days:
P10  |  P50  |  P90
0.2042 | 0.2600 | 0.3461
0.2272 | 0.2864 | 0.3826
0.2405 | 0.3360 | 0.4443
0.1517 | 0.1967 | 0.2576
0.1572 | 0.1796 | 0.2163
